# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. мы скорее получим dictionary с количестом букв в массиве: {"A" : 2, "B":1}. get ищет значение по ключу, а если в словаре нет ключа, то он выдаст 0, потому что у нас изначально dict r пустой и мы писали (с,0)-значит заменить на 0. А потом добавит туда +1, значит это уже идет как counter.
2. скорее ответ будет [0, 1, 1], потому что append добавляет в конец массива 1, а мы вызвали функцию дважды, значит в изначальный массив 1 добавится два раза
3. у меня есть два варианта ответа: возможно мы получим просто 3, потому что len() - считает длину и возможно выйдет количство элементов в словаре, возможно даже выйдет 2, так как алматы встречается дважды в одинакововм формате (lower()) и он посчитается как 1 элемент
4. наверное мы получим словарь с элементами массива как ключ и длиной каждого элемента в массиве как его значение. {"ab":2, "abc":3, "ab":2}, потому что k:len(k) в цикле сохроняет как ключ(сам элемент массива):значние(длина элемента)
5. скорее мы получим 5, потому что сам по себе x - это вызов функции g(n), а если вместо n примянить 5 - то результат будет 5

In [6]:
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)

{'A': 2, 'B': 1}
[0, 1, 1]
2
{'ab': 2, 'abc': 3}
5
None


In [7]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [8]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [32]:
# База
#1
def clean_city(s):
  clean = s.strip().title()
  d = {"Almaty" : "Алматы", "Astana" : "Астана", "Shymkent" : "Шымкент", "Karaganda" : "Караганда", "Aktobe" : "Актобе", "Atyrau" : "Атырау"}
  return d.get(clean, clean)

print(clean_city(" Алматы"), clean_city("АЛМАТЫ"), clean_city("Shymkent"))

def to_num(s):
  num = s.replace(" ", "")
  return int(float(num))

print(to_num("8070.0"), to_num("29 340"))

#2
print("\n")
d1 = {}
for row in rows:
  d1[row["category"]] = d1.get(row["category"],0) + 1
result = sorted(d1.items(), key = lambda x: x[1], reverse = True)
print(d1)
print(result)

#3
print("\n")
revenue = {}
count = {}
avg = {}
for row in rows:
  if row["status"] == "оплачен":
    revenue[row['category']] = revenue.get(row['category'], 0) + to_num(row['amount'])
    count[row["category"]] = count.get(row["category"],0) + 1
    avg[row["category"]] = round(revenue[row['category']] / count[row["category"]], 2)
# print(revenue)
# print(count)
# print(avg)
print(f"{'Категория':<12} {'Выручка':>12} {'Заказов':>8} {'Средний чек':>12}")
for cat in revenue:
    print(f"{cat:<12} {revenue[cat]:>12,} {count[cat]:>8} {avg[cat]:>12,}")

#4
print("\n")
months = {}
for row in rows:
    month = row["order_date"][:7]
    months[month] = months.get(month, 0) + 1

for month, cnt in sorted(months.items()):
    print(f"{month:<10} {cnt:>6}")

Алматы Алматы Шымкент
8070 29340


{'Одежда': 1529, 'Красота': 991, 'Дом': 1105, 'Книги': 653, 'Электроника': 1185}
[('Одежда', 1529), ('Электроника', 1185), ('Дом', 1105), ('Красота', 991), ('Книги', 653)]


Категория         Выручка  Заказов  Средний чек
Одежда         18,321,080     1379    13,285.77
Красота         5,507,620      890     6,188.34
Дом            10,516,320      987    10,654.83
Книги           2,294,100      583     3,934.99
Электроника    65,270,240     1065    61,286.61


2026-09      2782
2026-10      2681


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [38]:
# Стандарт
#5
rev_month = {}
for row in rows:
    if row["status"] == "оплачен":
        month = row["order_date"][:7]
        rev_month[month] = rev_month.get(month, 0) + to_num(row["amount"])

for month, rev in sorted(rev_month.items()):
    print(f"{month:<10} {rev:>14,}")

sep = rev_month["2026-09"]
oct_ = rev_month["2026-10"]
change = (oct_ - sep) / sep * 100
print(f"Изменение октября к сентябрю: {change:.1f}%\n")

#6
total = {}
canceled = {}
for row in rows:
    city = clean_city(row["city"])
    total[city] = total.get(city, 0) + 1
    if row["status"] == "отменён":
        canceled[city] = canceled.get(city, 0) + 1

print(f"{'Город':<12} {'Всего':>6} {'Отменено':>9} {'Доля':>7}")
for city in total:
    c = canceled.get(city, 0)
    share = c / total[city] * 100
    print(f"{city:<12} {total[city]:>6} {c:>9} {share:>6.1f}%")

#7
print("\n")
rev_client = {}
for row in rows:
    if row["status"] == "оплачен":
        cid = row["customer_id"]
        rev_client[cid] = rev_client.get(cid, 0) + to_num(row["amount"])

top5 = sorted(rev_client.items(), key=lambda x: x[1], reverse=True)[:5]
for cid, rev in top5:
    print(f"{cid:<8} {rev:>12,}")
print("\n")

#8
clients = {}
for row in rows:
    city = clean_city(row["city"])
    if city not in clients:
        clients[city] = set()
    clients[city].add(row["customer_id"])

for city, ids in clients.items():
    print(f"{city:<12} {len(ids):>5}")

2026-09        57,489,850
2026-10        44,419,510
Изменение октября к сентябрю: -22.7%

Город         Всего  Отменено    Доля
Алматы         2188       172    7.9%
Астана         1390       123    8.8%
Караганда       538        48    8.9%
Шымкент         804       164   20.4%
Атырау          146        11    7.5%
Актобе          397        41   10.3%


TEST       29,999,970
C01627        257,180
C00485        215,920
C01338        192,080
C02122        191,670


Алматы        1351
Астана         981
Караганда      451
Шымкент        655
Атырау         138
Актобе         358


## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [ ]:
# Челлендж